# Seminar 9: Building HTTP applications

Today we will be building an HTTP application for a Q&A platform.

A speaker is presenting, and the audience can ask questions and vote on the questions that they like. The questions can be shown in a projector in order of most voted, and the speaker answer questions or delete them.

## API: Application Programming Interface

A contract between the "server" and the "client"

We need to agree on:

1. What data is exchanged, what the "server" knows, what the "client" knows
2. How the data is transmitted (protocol)

Protocols:

1. HTTP/1.1 (1997)
1. HTTP/2 (2015)
1. HTTP/3 (2022)
1. WebSocket (2011)

...

## HTTP Protocol

Key characteristics:
1. Interaction through verbs — GET, POST, PUT, DELETE, ...
2. The server stores no client state between requests (no sessions, no operation history. This should be handled by the client and the application state)

More details: https://developer.mozilla.org/en-US/docs/Web/HTTP/Overview

<img src="images/request_response.png" width="900">

## How an HTTP application is organised

<img src="images/http_app_layers.png" width="550">

A request goes down through the parts and the response comes back up. Each part has one concern:

- **Presentation:** handles HTTP: URLs, methods, status codes, input and output formats.
- **Domain:** the business rules.
- **Infrastructure:** talks to the outside world: storage (memory, a file, a database), caches, external APIs.

This is a general pattern, not a strict rule: small apps merge parts. In this seminar we skip the domain layer, so the routes talk to the storage directly.

## Frameworks

There are many different libraries and frameworks for writing REST APIs.                           
Here are some of them:
1. AIOHTTP
2. Django, Django Rest
3. Flask
4. Falcon
5. FastAPI

Here let's go with FastAPI because:
1. Supports asynchrony
2. Easy to test
3. Generates documentation out of the box

<img src="https://fastapi.tiangolo.com/img/logo-margin/logo-teal.png" width=500px>

Launch command:
```
uvicorn main:app --reload
```

`reload` — automatically reload the server when the code is changed.

Uvicorn is an ASGI (Asynchronous Server Gateway Interface) server implementation.

It handles the HTTP requests and responses, and runs the FastAPI application.

Uvicorn runs one asyncio event loop per worker process, and each request is a coroutine on that loop.

In [5]:
from importlib.resources import files

from fastapi import FastAPI
from fastapi.responses import FileResponse

app = FastAPI(title="AskBoard")


@app.get("/board", include_in_schema=False)
def board():
    """The client: a page that only calls our API."""
    return FileResponse(files("askboard") / "static" / "board.html")

In [6]:
# Notebook only: runs `app` like `uvicorn askboard.main:app --reload`,
# so the cells below can keep adding and removing routes to the running server.
from notebook_tools import client, run_in_background, show, simulate_users, token_for

run_in_background(app)

ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


KeyboardInterrupt: 

The client: **http://127.0.0.1:8000/board**

Some properties
* Safety: A safe method has no side effects. Does not change the state of the resource.

* Idempotency: An idempotent method does not change the state of the resource when the same request is repeated.

* Cacheability: A cacheable method allows the client and intermediate proxies to cache the result of a request.

Let's first define the data model.

In [7]:
from datetime import datetime

from pydantic import BaseModel, ConfigDict


class QuestionIn(BaseModel):
    """What a client sends to ask a question."""

    text: str
    author: str = "anonymous"
    author_email: str | None = None


class QuestionOut(BaseModel):
    """What a client gets back. The author's email stays on the server."""

    model_config = ConfigDict(from_attributes=True)

    id: int
    text: str
    author: str
    votes: int
    answered: bool
    created_at: datetime

We use Pydantic models: classes whose type-annotated attributes define the shape of the data. FastAPI reads the request body into `QuestionIn`, and sends the response through `QuestionOut`, which leaves out the fields it doesn't list, such as the email.

resources are identified by URLs: `/questions` is the collection, `/questions/{id}` is one question.

In [8]:
from fastapi import HTTPException, status

from askboard.storage import InMemoryQuestionStore

# keeps the questions and votes in dicts.
store = InMemoryQuestionStore()


def found(question):
    if question is None:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "Question not found")
    return question


@app.post("/questions", status_code=status.HTTP_201_CREATED, response_model=QuestionOut)
def ask_question(body: QuestionIn):
    return store.add(body.text, body.author, body.author_email)


@app.get("/questions", response_model=list[QuestionOut])
def read_questions():
    return store.list()


@app.get("/questions/{question_id}", response_model=QuestionOut)
def read_question(question_id: int):
    return found(store.get(question_id))

`GET`

* Returns a resource (more formally, its representation)
* Safe, idempotent, cacheable

`POST`

* Sends data to the server; can create a new resource with its own URL
* Unsafe, non-idempotent, non-cacheable

In [9]:
show(client.post("/questions", json={
    "text": "What does idempotent mean?",
    "author": "ana",
    "author_email": "ana@example.com",
}))

show(client.post("/questions", json={
    "text": "Why PUT and not POST?",
    "author": "ben",
}))

POST /questions
{
  "text": "What does idempotent mean?",
  "author": "ana",
  "author_email": "ana@example.com"
}
-> 201 Created
{
  "id": 1,
  "text": "What does idempotent mean?",
  "author": "ana",
  "votes": 0,
  "answered": false,
  "created_at": "2026-11-12T22:49:21.978378Z"
}

POST /questions
{
  "text": "Why PUT and not POST?",
  "author": "ben"
}
-> 201 Created
{
  "id": 2,
  "text": "Why PUT and not POST?",
  "author": "ben",
  "votes": 0,
  "answered": false,
  "created_at": "2026-11-12T22:49:21.980751Z"
}



In [10]:
show(client.get("/questions"))

show(client.get("/questions/1"))

# A question that does not exist, an id that is not a number
show(client.get("/questions/99"))

show(client.get("/questions/abc"))

GET /questions
-> 200 OK
[
  {
    "id": 2,
    "text": "Why PUT and not POST?",
    "author": "ben",
    "votes": 0,
    "answered": false,
    "created_at": "2026-11-12T22:49:21.980751Z"
  },
  {
    "id": 1,
    "text": "What does idempotent mean?",
    "author": "ana",
    "votes": 0,
    "answered": false,
    "created_at": "2026-11-12T22:49:21.978378Z"
  }
]

GET /questions/1
-> 200 OK
{
  "id": 1,
  "text": "What does idempotent mean?",
  "author": "ana",
  "votes": 0,
  "answered": false,
  "created_at": "2026-11-12T22:49:21.978378Z"
}

GET /questions/99
-> 404 Not Found
{
  "detail": "Question not found"
}

GET /questions/abc
-> 422 Unprocessable Entity
{
  "detail": [
    {
      "type": "int_parsing",
      "loc": [
        "path",
        "question_id"
      ],
      "msg": "Input should be a valid integer, unable to parse string as an integer",
      "input": "abc"
    }
  ]
}



In [11]:
# Invalid questions: only spaces, too long
show(client.post("/questions", json={"text": "        "}))

show(client.post("/questions", json={"text": "?" * 1000}))

POST /questions
{
  "text": "        "
}
-> 201 Created
{
  "id": 3,
  "text": "        ",
  "author": "anonymous",
  "votes": 0,
  "answered": false,
  "created_at": "2026-11-12T22:49:23.022169Z"
}

POST /questions
{
  "text": "????????????????????????????????????????????????????????????????????????????????... (1000 characters)"
}
-> 201 Created
{
  "id": 4,
  "text": "????????????????????????????????????????????????????????????????????????????????... (1000 characters)",
  "author": "anonymous",
  "votes": 0,
  "answered": false,
  "created_at": "2026-11-12T22:49:23.024181Z"
}



We need to validate certain properties when modeling the data. For that we use Pydantic.

In [12]:
from datetime import datetime

from pydantic import BaseModel, ConfigDict, Field, field_validator


class QuestionIn(BaseModel):
    """What a client sends to ask a question."""

    text: str = Field(min_length=3, max_length=280, examples=["What does idempotent mean?"])
    author: str = Field("anonymous", max_length=40)
    author_email: str | None = None

    @field_validator("text")
    @classmethod
    def not_only_spaces(cls, text: str) -> str:
        if len(text.strip()) < 3:
            raise ValueError("Not enough content")
        return text.strip()


class QuestionOut(BaseModel):
    """What a client gets back. The author's email stays on the server."""

    model_config = ConfigDict(from_attributes=True)

    id: int
    text: str
    author: str
    votes: int
    answered: bool
    created_at: datetime

In [13]:
@app.post("/questions", status_code=status.HTTP_201_CREATED, response_model=QuestionOut)
def ask_question(body: QuestionIn):
    return store.add(body.text, body.author, body.author_email)

In [14]:
# The same invalid questions as before
show(client.post("/questions", json={"text": "        "}))

show(client.post("/questions", json={"text": "?" * 1000}))

POST /questions
{
  "text": "        "
}
-> 201 Created
{
  "id": 5,
  "text": "        ",
  "author": "anonymous",
  "votes": 0,
  "answered": false,
  "created_at": "2026-11-12T22:49:25.015355Z"
}

POST /questions
{
  "text": "????????????????????????????????????????????????????????????????????????????????... (1000 characters)"
}
-> 201 Created
{
  "id": 6,
  "text": "????????????????????????????????????????????????????????????????????????????????... (1000 characters)",
  "author": "anonymous",
  "votes": 0,
  "answered": false,
  "created_at": "2026-11-12T22:49:25.017685Z"
}



When an instance is created, Pydantic converts and checks every field, and raises a `ValidationError` listing all the problems; FastAPI turns that into a `422`. `Field` attaches constraints and documentation to an attribute:

| Parameter | For | Example |
|---|---|---|
| `default` | value when the field is missing | `Field("anonymous")` |
| `min_length`, `max_length` | strings, lists | `Field(min_length=3)` |
| `gt`, `ge`, `lt`, `le` | numbers | `Field(ge=0)` |
| `pattern` | strings, a regular expression | `Field(pattern=r"^[a-z]+$")` |
| `description`, `examples` | shown in `/docs` | `Field(examples=["..."])` |

For a rule `Field` can't express, write a function with `@field_validator`; raising `ValueError` there gives the same `422`.

In [15]:
from pydantic import ValidationError

# dict -> model: parse and validate
question = QuestionIn.model_validate({"text": "What is REST?", "author": "ana"})
print(repr(question))

# model -> dict (what goes out as JSON)
print(question.model_dump())

# The same with keyword arguments
print(repr(QuestionIn(text="What is HTTP?")))

# Invalid data
try:
    QuestionIn.model_validate({"text": "  ", "author": "x" * 50})
except ValidationError as error:
    print(error)

# Any object with the same attributes -> model (from_attributes): what response_model does
print(repr(QuestionOut.model_validate(store.get(1))))

QuestionIn(text='What is REST?', author='ana', author_email=None)
{'text': 'What is REST?', 'author': 'ana', 'author_email': None}
QuestionIn(text='What is HTTP?', author='anonymous', author_email=None)
2 validation errors for QuestionIn
text
  String should have at least 3 characters [type=string_too_short, input_value='  ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_short
author
  String should have at most 40 characters [type=string_too_long, input_value='xxxxxxxxxxxxxxxxxxxxxxxx...xxxxxxxxxxxxxxxxxxxxxxx', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_long
QuestionOut(id=1, text='What does idempotent mean?', author='ana', votes=0, answered=False, created_at=datetime.datetime(2026, 11, 12, 22, 49, 21, 978378, tzinfo=datetime.timezone.utc))


Open **http://127.0.0.1:8000/docs**: the documentation is generated from the code. The board can now ask and list.

In [ ]:
simulate_users(questions=4)

---

Now let's make the audience able to upvote a question.

**How do we represent a vote as a new resource?**

what is the problem with `POST /questions/{id}/votes`?

In [ ]:
# some space before the answer :)












@app.put("/questions/{question_id}/votes/{user}", response_model=QuestionOut)
def vote(question_id: int, user: str):
    return found(store.vote(question_id, user))

In [ ]:
# The same vote, five times
for _ in range(5):
    response = client.put("/questions/1/votes/ana")
show(response)

show(client.put("/questions/99/votes/ana"))

simulate_users(votes=15)

`PUT`

* Can either update the current resource or create a new one if it does not exist
* Unsafe, idempotent, non-cacheable

Now the speaker should be able to mark a question as answered, and delete one.

**Which methods can we use for that?**

In [ ]:
# :O














class QuestionUpdate(BaseModel):
    answered: bool


@app.patch("/questions/{question_id}", response_model=QuestionOut)
def update_question(question_id: int, body: QuestionUpdate):
    return found(store.set_answered(question_id, body.answered))


@app.delete("/questions/{question_id}", status_code=status.HTTP_204_NO_CONTENT)
def remove_question(question_id: int):
    if not store.delete(question_id):
        raise HTTPException(status.HTTP_404_NOT_FOUND, "Question not found")

In [ ]:
show(client.patch("/questions/1", json={"answered": True}))

# Delete twice, then read it
show(client.delete("/questions/2"))

show(client.delete("/questions/2"))

show(client.get("/questions/2"))

`PATCH`

* Partially modifies a resource: send only the part that changes
* Unsafe, possibly non-idempotent, non-cacheable

`DELETE`

* Deletes a resource by URL (the inverse of `PUT`)
* Unsafe, idempotent, non-cacheable

Relationship between `GET`, `PUT`, `DELETE`.

* If `PUT` was executed, then `GET` must return the same representation
* If `DELETE` was executed, then `GET` must return an error

Now we need that the most voted questions appear on top and that the answered ones are filtered out.

In [ ]:
from typing import Literal

from fastapi import Query


@app.get("/questions", response_model=list[QuestionOut])
def read_questions(
    sort: Literal["recent", "votes"] = "recent",
    limit: int = Query(20, ge=1, le=100),
    offset: int = Query(0, ge=0),
    answered: bool | None = None,
):
    return store.list(sort, limit, offset, answered)

In [ ]:
show(client.get("/questions", params={"sort": "votes", "limit": 3}))

show(client.get("/questions", params={"answered": True}))

# Out of range: limit is at most 100
show(client.get("/questions", params={"limit": 1000}))

#### QUERY method

queries in complex systems can exceed the URL length limit. The new HTTP `QUERY` method ([RFC 10008](https://www.rfc-editor.org/rfc/rfc10008.html)) is safe and idempotent like `GET`, but includes the body in the query.

Now how if the users and the speaker use the same client, how do determine who can mark and delete questions?

Representing the user as a name string is also a problem, what if there are:

- 2 people with the same name?
- Malicious spammers?

Authorization and authentication are generally needed in a web app.

In [ ]:
import secrets
from dataclasses import dataclass
from typing import Annotated

from fastapi import Depends
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer

SPEAKER_PASSWORD = "speaker"
Role = Literal["audience", "speaker"]


class LoginIn(BaseModel):
    """No password: audience. The speaker password: speaker."""

    name: str = Field(min_length=1, max_length=40)
    password: str | None = None


class TokenOut(BaseModel):
    access_token: str
    token_type: Literal["bearer"] = "bearer"
    name: str
    role: Role


@dataclass(frozen=True)
class User:
    name: str
    role: Role


sessions: dict[str, User] = {}  # token -> user


@app.post("/login", response_model=TokenOut)
def login(body: LoginIn):
    if body.password is None:
        role = "audience"
    elif secrets.compare_digest(body.password, SPEAKER_PASSWORD):
        role = "speaker"
    else:
        raise HTTPException(status.HTTP_401_UNAUTHORIZED, "Wrong password")
    token = secrets.token_urlsafe(24)
    sessions[token] = User(body.name, role)
    return TokenOut(access_token=token, name=body.name, role=role)


bearer = HTTPBearer(auto_error=False)


def current_user(credentials: Annotated[HTTPAuthorizationCredentials | None, Depends(bearer)]) -> User:
    user = sessions.get(credentials.credentials) if credentials else None
    if user is None:
        raise HTTPException(status.HTTP_401_UNAUTHORIZED, "Not logged in", headers={"WWW-Authenticate": "Bearer"})
    return user


def speaker(user: Annotated[User, Depends(current_user)]) -> User:
    if user.role != "speaker":
        raise HTTPException(status.HTTP_403_FORBIDDEN, "Only the speaker can do this")
    return user


CurrentUser = Annotated[User, Depends(current_user)]
Speaker = Annotated[User, Depends(speaker)]

Bearer token: a identifier that the server or an authorization server(Oauth2) hands out at login. The client sends it in a header with every request

Authorization: Bearer <token>

In [ ]:
class QuestionIn(BaseModel):
    text: str = Field(min_length=3, max_length=280, examples=["What does idempotent mean?"])
    author_email: str | None = None

    @field_validator("text")
    @classmethod
    def not_only_spaces(cls, text: str) -> str:
        if len(text.strip()) < 3:
            raise ValueError("write a real question, not just spaces")
        return text.strip()


@app.post("/questions", status_code=status.HTTP_201_CREATED, response_model=QuestionOut)
def ask_question(body: QuestionIn, user: CurrentUser):
    return store.add(body.text, user.name, body.author_email)


@app.put("/questions/{question_id}/vote", response_model=QuestionOut)
def vote(question_id: int, user: CurrentUser):
    return found(store.vote(question_id, user.name))


@app.patch("/questions/{question_id}", response_model=QuestionOut)
def update_question(question_id: int, body: QuestionUpdate, _: Speaker):
    return found(store.set_answered(question_id, body.answered))


@app.delete("/questions/{question_id}", status_code=status.HTTP_204_NO_CONTENT)
def remove_question(question_id: int, _: Speaker):
    if not store.delete(question_id):
        raise HTTPException(status.HTTP_404_NOT_FOUND, "Question not found")

In [ ]:
ana = token_for("ana")
eva = token_for("eva", SPEAKER_PASSWORD)

# Not logged in, then logged in
show(client.post("/questions", json={"text": "Who am I?"}))

show(client.post("/questions", json={"text": "Who am I?"}, headers=ana))

show(client.put("/questions/1/vote", headers=ana))

# Audience, then speaker
show(client.delete("/questions/3", headers=ana))

show(client.delete("/questions/3", headers=eva))

`401 Unauthorized` -> "who are you???",

`403 Forbidden` -> "It's you ...  no you can't".

We have so far a functional Q&A board.

Some possible extensions are:
- **Rate limiting:** Prevent someone from taking down your server by sending too many requests or spam.
- **Caching**: If nothing changed on the board don't process the get request and send a cached response.

---

Now the server restarts (a deploy, a crash). What happens to the questions?

In [ ]:
# What a restart does to everything kept in memory:
store = InMemoryQuestionStore()
sessions.clear()

This is only one of the possible problems; JSON file storage would solve that one.

| Problem | Memory | JSON file | Database |
|---|---|---|---|
| Data survives a restart | No | Yes | Yes |
| Several requests write at once | Needs our lock | Writers overwrite each other's file | Transactions |
| A crash halfway through a write | Lost | The file can be left broken | All or nothing |
| Several worker processes | Each has its own copy | Shared, but not coordinated | Shared and coordinated |
| Rules like "one vote per person" | Our code | Our code | Constraints (`UNIQUE`, `NOT NULL`, foreign keys) |
| Sort, filter, count | Load everything into Python | Read the whole file | A query language, with indexes |

## Database

A relational database (RDB) is a type of database that stores data in tables (also called relations), where each table consists of rows (records) and columns (fields). The relationships between tables are defined using keys.

Key idea: Data is structured, organized, and linked, making it easy to retrieve and maintain.

- A **row** is one entity (a question).
- A **column** is one attribute of it, with a type (`INTEGER`, `TEXT`, ...).
- The **primary key (PK)** is a column that identifies each row: unique and never `NULL`. In Table 1 it is `id`.
- The **Foreign Key (FK)** is a field that links entities from one table to another.


<img src="images/questions_table.png" width="800">

Table 1. The `questions` table

### SQLite
SQLite is a relational database that lives in a single file, with no server to install or run. Comes in python standard library and is a good choice for small apps, testing and learning.

Production systems use a database server such as PostgreSQL, shared by many app processes.

In [ ]:
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

Path("seminar.db").unlink(missing_ok=True)  # start from an empty database

con = sqlite3.connect("seminar.db")
con.execute("PRAGMA foreign_keys = ON")  # off by default in SQLite

con.execute("""
CREATE TABLE questions (
    id           INTEGER PRIMARY KEY,
    text         TEXT    NOT NULL,
    author       TEXT    NOT NULL,
    author_email TEXT,
    answered     INTEGER NOT NULL DEFAULT 0,
    created_at   TEXT    NOT NULL
)""")
print("questions table created")

The two queries we need, writing and reading:

```sql
INSERT INTO table (column1, column2) VALUES (?, ?)

SELECT column1, column2 FROM table WHERE condition ORDER BY column LIMIT n

DELETE FROM table WHERE condition
```

The `?` placeholders are filled with values passed separately.

In [ ]:
def add_question(conn, text, author, author_email=None):
    conn.execute(
        "INSERT INTO questions (text, author, author_email, created_at) VALUES (?, ?, ?, ?)",
        (text, author, author_email, datetime.now(timezone.utc).isoformat()),
    )
    conn.commit()


def get_questions(conn):
    return conn.execute("SELECT id, text, author FROM questions ORDER BY id").fetchall()


add_question(con, "What does idempotent mean?", "ana", "ana@example.com")
add_question(con, "Why PUT and not POST?", "ben")
add_question(con, "How do I version an API?", "carla")
get_questions(con)

**Commit:** what a connection writes stays pending until `commit()`. Then it is saved to the file (or server) and other connections can see it

`rollback()` throws the pending changes away instead.

In [ ]:
# The schema says text is NOT NULL: the database rejects it
try:
    add_question(con, None, "ana")
except sqlite3.IntegrityError as error:
    print("IntegrityError:", error)
    con.rollback()  # discard the failed insert

<img src="images/questions_votes_erd.png" width="400">

Table 2. `votes.question_id` refers to `questions.id`

In [ ]:
con.execute("""
CREATE TABLE votes (
    question_id INTEGER NOT NULL REFERENCES questions (id) ON DELETE CASCADE,  -- deleting a question deletes its votes
    voter       TEXT    NOT NULL,
    UNIQUE (question_id, voter)
)""")


def add_vote(conn, question_id, voter):
    conn.execute("INSERT INTO votes (question_id, voter) VALUES (?, ?)", (question_id, voter))
    conn.commit()


add_vote(con, 1, "ben")
add_vote(con, 1, "carla")
add_vote(con, 3, "ana")

# The same person votes twice
try:
    add_vote(con, 1, "ben")
except sqlite3.IntegrityError as error:
    print("IntegrityError:", error)
    con.rollback()

Now we want to count the votes per question. This query requires aggregation over data in different tables.
For this we use the 'JOIN' operation

In [ ]:
con.execute("""
    SELECT q.id, q.text, COUNT(v.voter) AS votes
    FROM questions AS q
    LEFT JOIN votes AS v ON v.question_id = q.id
    GROUP BY q.id
    ORDER BY votes DESC
""").fetchall()

A join combines rows from two tables into one result. Each row of the left table is paired with every row of the right table that satisfies the ON condition, usually the foreign key equals the primary key (v.question_id = q.id). Each pair becomes one result row with the columns of both.

### Transactions

In [ ]:
from contextlib import contextmanager

@contextmanager
def connect(db_path):
    """One transaction: commits on success, rolls back on error, always closes."""
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row  # rows can be read by column name
    conn.execute("PRAGMA foreign_keys = ON")
    try:
        with conn:  # commit, or rollback if an exception was raised
            yield conn
    finally:
        conn.close()

In [ ]:
# A question and a vote in one transaction. The second fails, we roll back.
try:
    with connect("seminar.db") as conn:
        conn.execute(
            "INSERT INTO questions (text, author, created_at) VALUES (?, ?, ?)",
            ("Half of a transaction", "finn", datetime.now(timezone.utc).isoformat()),
        )
        conn.execute("INSERT INTO votes (question_id, voter) VALUES (1, 'ben')")
except sqlite3.IntegrityError as error:
    print("IntegrityError:", error)

get_questions(con)

A **transaction** groups statements into one unit:

- **Atomicity:** all of them are saved, or none (on an error, everything is rolled back).
- **Isolation:** other connections don't see the changes until the commit.

That is why every method of our SQLite store runs inside `connect()`: for example, a vote checks that the question exists and inserts the vote in one transaction.

Now we can swap our in-memory store for a SQLite one, and the questions will survive a server restart.

In [ ]:
from askboard.storage import SQLiteQuestionStore

store = SQLiteQuestionStore("askboard.db")

In [ ]:
author = "x' OR '1'='1"

# Built with string formatting
print(con.execute(f"SELECT id, text FROM questions WHERE author = '{author}'").fetchall())

# With a placeholder
print(con.execute("SELECT id, text FROM questions WHERE author = ?", (author,)).fetchall())

Avoid building a query with string formatting (f-strings, `+`, `%`). A value that contains SQL becomes part of the query: this is **SQL injection**.



To save us the effort of building these queries by hand, mapping rows to objects and back, and generating the SQL for different databases, we can use a OOP representation of the database.

### ORM (SQLAlchemy)

An **ORM** (object-relational mapper) maps tables to Python classes and rows to objects. You write queries in Python, and the ORM generates the SQL for your database, always with placeholders. SQLAlchemy is the most used ORM in Python:

```python
class Question(Base):
    __tablename__ = "questions"

    id: Mapped[int] = mapped_column(primary_key=True)
    text: Mapped[str]
    answered: Mapped[bool] = mapped_column(default=False)


with Session(engine) as session:
    open_questions = session.scalars(select(Question).where(Question.answered == False)).all()
```

As a next step, write a SQLAlchemy version of `SQLiteQuestionStore`: the routes don't change.